# Few-Shot Bearing Fault Diagnosis — Colab Runner

Self-contained notebook: writes out the full project source tree, installs pinned
dependencies, downloads the public CWRU bearing dataset, builds windowed features,
and runs the full label-fraction experiment grid (SVM/RF, 1D-CNN, LSTM, Prototypical
Networks, SimCLR contrastive pretraining) — all CPU-only, no GPU required.

**Recommended runtime:** Runtime > Change runtime type > CPU (GPU not required; the
code hardcodes `device: cpu` in the configs, though you can edit that to `cuda` if a
GPU runtime is available — it will speed up the CNN/ProtoNet/SimCLR methods).

**Persisting results across disconnects:** Colab's local disk is ephemeral. Run the
'Mount Google Drive' cell below and point `results/` there so a disconnect doesn't
lose your progress — the experiment runner checkpoints every completed run to
`results/tables/raw_results.jsonl` and **skips already-completed configs on rerun**,
so you can safely restart this notebook after any disconnect.

## 0. (Optional) Mount Google Drive for persistent results

In [ ]:
# Uncomment to persist results/ across Colab disconnects.
# from google.colab import drive
# drive.mount('/content/drive')
# import os
# os.makedirs('/content/drive/MyDrive/few-shot-bearing-fault-diagnosis', exist_ok=True)
# %cd /content/drive/MyDrive/few-shot-bearing-fault-diagnosis


## 1. Create project directory structure

In [ ]:
import os

dirs = [
    'configs', 'data/raw', 'data/processed', 'results/figures',
    'results/tables', 'results/models',
    'src', 'src/data', 'src/features', 'src/models', 'src/self_supervised',
    'src/evaluation', 'src/utils', 'tests',
]
for d in dirs:
    os.makedirs(d, exist_ok=True)
print('Directory structure created.')


## 2. Write out all project source files

In [ ]:
%%writefile requirements.txt
numpy==1.26.4
pandas==2.2.1
scipy==1.12.0
scikit-learn==1.4.1.post1
torch==2.2.1
matplotlib==3.8.3
seaborn==0.13.2
pyyaml==6.0.1
pytest==8.1.1
requests==2.31.0
tqdm==4.66.2
statsmodels==0.14.1


In [ ]:
%%writefile configs/default.yaml
# Default experiment configuration for few-shot bearing fault diagnosis.

data:
  raw_dir: data/raw/cwru
  processed_path: data/processed/windows.npz
  window_size: 1024
  stride: 512
  channel: DE
  sampling_rate_hz: 12000

experiment:
  label_fractions: [0.05, 0.10, 0.25, 0.50, 1.00]
  seeds: [0, 1, 2, 3, 4]
  test_size: 0.3
  methods:
    - svm
    - random_forest
    - cnn1d
    - lstm
    - protonet
    - simclr_linear

cross_load:
  enabled: false
  train_loads: [0, 1, 2]
  test_loads: [3]

training:
  batch_size: 64
  epochs: 20
  learning_rate: 0.001
  weight_decay: 0.0001
  embedding_dim: 64
  device: cpu

lstm:
  # Plain LSTMs converge much slower than the CNN backbone (weaker inductive
  # bias for this task) and were observed to collapse to majority-class
  # prediction within the shared 20-epoch budget, especially at low label
  # fractions. Give it a larger, dedicated epoch budget.
  epochs: 80

protonet:
  n_way: 10
  n_support: 5
  n_query: 15
  episodes_per_epoch: 50
  epochs: 10

simclr:
  epochs: 10
  batch_size: 64
  temperature: 0.5
  projection_dim: 32
  linear_probe_epochs: 40

masked_reconstruction:
  epochs: 40
  batch_size: 64
  mask_ratio: 0.15
  span_len: 32

output:
  results_dir: results
  figures_dir: results/figures
  tables_dir: results/tables
  models_dir: results/models


In [ ]:
%%writefile configs/cross_load.yaml
# Cross-load generalization experiment: train on loads 0-2 HP, test on
# held-out load 3 HP, to evaluate robustness beyond same-condition overfitting.

data:
  raw_dir: data/raw/cwru
  processed_path: data/processed/windows.npz
  window_size: 1024
  stride: 512
  channel: DE
  sampling_rate_hz: 12000

experiment:
  label_fractions: [0.05, 0.10, 0.25, 0.50, 1.00]
  seeds: [0, 1, 2, 3, 4]
  methods:
    - svm
    - random_forest
    - cnn1d
    - lstm
    - protonet
    - simclr_linear

cross_load:
  enabled: true
  train_loads: [0, 1, 2]
  test_loads: [3]

training:
  batch_size: 64
  epochs: 20
  learning_rate: 0.001
  weight_decay: 0.0001
  embedding_dim: 64
  device: cpu

lstm:
  epochs: 80

protonet:
  n_way: 10
  n_support: 5
  n_query: 15
  episodes_per_epoch: 50
  epochs: 10

simclr:
  epochs: 10
  batch_size: 64
  temperature: 0.5
  projection_dim: 32
  linear_probe_epochs: 40

output:
  results_dir: results/cross_load
  figures_dir: results/cross_load/figures
  tables_dir: results/cross_load/tables
  models_dir: results/cross_load/models


In [ ]:
%%writefile src/__init__.py


In [ ]:
%%writefile src/data/__init__.py


In [ ]:
%%writefile src/data/preprocessing.py
"""Signal preprocessing: normalization and sliding-window segmentation."""
from typing import Tuple

import numpy as np


def normalize_signal(signal: np.ndarray) -> np.ndarray:
    """Zero-mean, unit-variance normalization of a 1D signal."""
    signal = np.asarray(signal, dtype=np.float64)
    std = signal.std()
    if std < 1e-12:
        return signal - signal.mean()
    return (signal - signal.mean()) / std


def sliding_window(
    signal: np.ndarray, window_size: int, stride: int
) -> np.ndarray:
    """Segment a 1D signal into overlapping windows.

    Returns an array of shape (n_windows, window_size). Any trailing samples
    that don't fill a full window are dropped.
    """
    signal = np.asarray(signal)
    if window_size <= 0 or stride <= 0:
        raise ValueError("window_size and stride must be positive")
    n = len(signal)
    if n < window_size:
        return np.empty((0, window_size), dtype=signal.dtype)
    n_windows = (n - window_size) // stride + 1
    windows = np.empty((n_windows, window_size), dtype=signal.dtype)
    for i in range(n_windows):
        start = i * stride
        windows[i] = signal[start : start + window_size]
    return windows


def windows_and_labels(
    signal: np.ndarray, label: int, window_size: int, stride: int
) -> Tuple[np.ndarray, np.ndarray]:
    """Window a signal and produce a matching label array."""
    windows = sliding_window(signal, window_size, stride)
    labels = np.full(len(windows), label, dtype=np.int64)
    return windows, labels


In [ ]:
%%writefile src/data/cwru.py
"""CWRU Bearing Dataset loading utilities.

Loads the official .mat files from the Case Western Reserve University
Bearing Data Center. Files are expected under `data/raw/cwru/` after
running `src/data/download_cwru.py`.

Each .mat file contains one or more channels named like:
  X097_DE_time, X097_FE_time, X097_BA_time, X097RPM
The prefix number is the file id; DE = drive end accelerometer,
FE = fan end accelerometer, BA = base accelerometer.
"""
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
from scipy.io import loadmat

# Fault class labels used throughout the project.
CLASS_NAMES = [
    "normal",
    "ball_007",
    "ball_014",
    "ball_021",
    "inner_race_007",
    "inner_race_014",
    "inner_race_021",
    "outer_race_007",
    "outer_race_014",
    "outer_race_021",
]
CLASS_TO_IDX = {name: i for i, name in enumerate(CLASS_NAMES)}


@dataclass(frozen=True)
class CWRUFile:
    """Metadata for a single CWRU .mat recording."""

    file_id: str
    label: str
    load_hp: int  # motor load in HP (0-3)
    filename: str


def load_mat_signal(path: Path, channel: str = "DE") -> np.ndarray:
    """Load the specified accelerometer channel from a CWRU .mat file.

    channel: one of "DE", "FE", "BA".
    """
    mat = loadmat(str(path))
    key = None
    for k in mat.keys():
        if k.endswith(f"{channel}_time"):
            key = k
            break
    if key is None:
        raise KeyError(f"No {channel}_time channel found in {path}")
    return mat[key].squeeze().astype(np.float64)


def list_available_files(raw_dir: Path) -> List[Path]:
    raw_dir = Path(raw_dir)
    return sorted(raw_dir.glob("*.mat"))


def build_manifest(index: Dict[str, dict]) -> List[CWRUFile]:
    """Build a manifest of CWRUFile entries from an index dict.

    `index` maps filename -> {"label": str, "load_hp": int}.
    """
    manifest = []
    for filename, meta in index.items():
        manifest.append(
            CWRUFile(
                file_id=Path(filename).stem,
                label=meta["label"],
                load_hp=meta["load_hp"],
                filename=filename,
            )
        )
    return manifest


In [ ]:
%%writefile src/data/cwru_manifest.py
"""Manifest of the standard CWRU 12kHz Drive-End fault dataset subset used
in this project.

Subset: 12 kHz Drive End bearing fault data, fault diameters 0.007" /
0.014" / 0.021", centroid fault location OR@6:00, motor loads 0-3 HP
(approx. 1797/1772/1750/1730 RPM), plus normal baseline. This mirrors the
most commonly used CWRU benchmark subset in the fault-diagnosis literature.

File numbers below correspond to the official CWRU Bearing Data Center
file numbering (https://engineering.case.edu/bearingdatacenter).
Each entry maps: file_number -> (label, load_hp).
"""
from typing import Dict, Tuple

# label -> {load_hp: file_number}
CWRU_12K_DE_FILES: Dict[str, Dict[int, int]] = {
    "normal": {0: 97, 1: 98, 2: 99, 3: 100},
    "ball_007": {0: 118, 1: 119, 2: 120, 3: 121},
    "ball_014": {0: 185, 1: 186, 2: 187, 3: 188},
    "ball_021": {0: 222, 1: 223, 2: 225, 3: 226},
    "inner_race_007": {0: 105, 1: 106, 2: 107, 3: 108},
    "inner_race_014": {0: 169, 1: 170, 2: 171, 3: 172},
    "inner_race_021": {0: 209, 1: 210, 2: 211, 3: 212},
    "outer_race_007": {0: 130, 1: 131, 2: 132, 3: 133},
    "outer_race_014": {0: 197, 1: 198, 2: 199, 3: 200},
    "outer_race_021": {0: 234, 1: 235, 2: 236, 3: 237},
}


def build_index() -> Dict[str, dict]:
    """Return {filename: {"label": str, "load_hp": int}} for downloading."""
    index = {}
    for label, loads in CWRU_12K_DE_FILES.items():
        for load_hp, file_number in loads.items():
            filename = f"{file_number}.mat"
            index[filename] = {"label": label, "load_hp": load_hp, "file_number": file_number}
    return index


def file_number_to_url(file_number: int) -> str:
    """Construct the CWRU Bearing Data Center download URL for a file number.

    The Bearing Data Center serves files at a stable per-file URL of the
    form https://engineering.case.edu/sites/default/files/<id>.mat
    """
    return f"https://engineering.case.edu/sites/default/files/{file_number}.mat"


In [ ]:
%%writefile src/data/download_cwru.py
"""Download the CWRU 12kHz Drive-End bearing dataset subset.

No credentials/API key required — files are served publicly by the Case
Western Reserve University Bearing Data Center. Downloads are deterministic:
re-running this script re-fetches the exact same fixed set of file numbers
defined in `src/data/cwru_manifest.py`.

Usage:
    python -m src.data.download_cwru --out data/raw/cwru
"""
import argparse
import time
from pathlib import Path

import requests

from src.data.cwru_manifest import build_index, file_number_to_url


def download_file(url: str, dest: Path, retries: int = 3, timeout: int = 30) -> bool:
    if dest.exists() and dest.stat().st_size > 0:
        return True
    for attempt in range(retries):
        try:
            resp = requests.get(url, timeout=timeout)
            resp.raise_for_status()
            dest.write_bytes(resp.content)
            return True
        except requests.RequestException as e:
            print(f"  attempt {attempt + 1}/{retries} failed: {e}")
            time.sleep(2)
    return False


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--out", default="data/raw/cwru", help="Output directory")
    args = parser.parse_args()

    out_dir = Path(args.out)
    out_dir.mkdir(parents=True, exist_ok=True)

    index = build_index()
    print(f"Downloading {len(index)} CWRU .mat files to {out_dir} ...")
    failures = []
    for filename, meta in index.items():
        dest = out_dir / filename
        url = file_number_to_url(meta["file_number"])
        print(f"[{meta['label']} @ {meta['load_hp']}HP] {url} -> {dest.name}")
        ok = download_file(url, dest)
        if not ok:
            failures.append(filename)

    if failures:
        print(f"\nWARNING: {len(failures)} files failed to download: {failures}")
        print("Retry the script, or check https://engineering.case.edu/bearingdatacenter for mirrors.")
    else:
        print("\nAll files downloaded successfully.")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/data/build_dataset.py
"""Build windowed train/test arrays from raw CWRU .mat files.

Usage:
    python -m src.data.build_dataset --raw data/raw/cwru --out data/processed \
        --window-size 1024 --stride 512 --channel DE

Saves data/processed/windows.npz containing:
  X: (N, window_size) float32
  y: (N,) int64 class labels
  load_hp: (N,) int64 motor load in HP
  class_names: list of class name strings
"""
import argparse
from pathlib import Path

import numpy as np

from src.data.cwru import CLASS_NAMES, CLASS_TO_IDX, load_mat_signal
from src.data.cwru_manifest import build_index
from src.data.preprocessing import normalize_signal, windows_and_labels


def build(raw_dir: Path, window_size: int, stride: int, channel: str):
    index = build_index()
    all_X, all_y, all_load = [], [], []
    missing = []
    for filename, meta in index.items():
        path = raw_dir / filename
        if not path.exists():
            missing.append(filename)
            continue
        signal = load_mat_signal(path, channel=channel)
        signal = normalize_signal(signal)
        label_idx = CLASS_TO_IDX[meta["label"]]
        windows, labels = windows_and_labels(signal, label_idx, window_size, stride)
        all_X.append(windows)
        all_y.append(labels)
        all_load.append(np.full(len(windows), meta["load_hp"], dtype=np.int64))

    if missing:
        raise FileNotFoundError(
            f"{len(missing)} raw files missing (run download_cwru.py first): {missing[:5]}..."
        )

    X = np.concatenate(all_X, axis=0).astype(np.float32)
    y = np.concatenate(all_y, axis=0)
    load_hp = np.concatenate(all_load, axis=0)
    return X, y, load_hp


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--raw", default="data/raw/cwru")
    parser.add_argument("--out", default="data/processed")
    parser.add_argument("--window-size", type=int, default=1024)
    parser.add_argument("--stride", type=int, default=512)
    parser.add_argument("--channel", default="DE")
    args = parser.parse_args()

    raw_dir = Path(args.raw)
    out_dir = Path(args.out)
    out_dir.mkdir(parents=True, exist_ok=True)

    X, y, load_hp = build(raw_dir, args.window_size, args.stride, args.channel)
    out_path = out_dir / "windows.npz"
    np.savez_compressed(
        out_path, X=X, y=y, load_hp=load_hp, class_names=np.array(CLASS_NAMES)
    )
    print(f"Saved {X.shape[0]} windows of length {X.shape[1]} to {out_path}")
    print(f"Class distribution: {np.bincount(y)}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/data/splits.py
"""Train/test splitting utilities: label-fraction subsampling and
cross-load generalization splits."""
from typing import List, Tuple

import numpy as np
from sklearn.model_selection import train_test_split


def stratified_label_fraction_split(
    y: np.ndarray, fraction: float, seed: int
) -> np.ndarray:
    """Return indices of a stratified subsample of size `fraction` of y.

    Guarantees at least 1 sample per class when fraction > 0.
    """
    if not 0 < fraction <= 1.0:
        raise ValueError("fraction must be in (0, 1]")
    if fraction == 1.0:
        return np.arange(len(y))

    rng = np.random.RandomState(seed)
    indices = []
    for cls in np.unique(y):
        cls_idx = np.where(y == cls)[0]
        n_take = max(1, int(round(len(cls_idx) * fraction)))
        chosen = rng.choice(cls_idx, size=n_take, replace=False)
        indices.append(chosen)
    return np.sort(np.concatenate(indices))


def train_test_split_stratified(
    X: np.ndarray, y: np.ndarray, test_size: float, seed: int
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    return train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=seed
    )


def cross_load_split(
    load_hp: np.ndarray, train_loads: List[int], test_loads: List[int]
) -> Tuple[np.ndarray, np.ndarray]:
    """Return (train_idx, test_idx) for training on `train_loads` HP
    conditions and testing on held-out `test_loads` conditions."""
    train_idx = np.where(np.isin(load_hp, train_loads))[0]
    test_idx = np.where(np.isin(load_hp, test_loads))[0]
    return train_idx, test_idx


In [ ]:
%%writefile src/features/__init__.py


In [ ]:
%%writefile src/features/handcrafted.py
"""Hand-crafted time-domain and spectral features for classical baselines
(SVM / Random Forest)."""
import numpy as np
from scipy.fft import rfft, rfftfreq
from scipy.stats import kurtosis, skew


def time_domain_features(window: np.ndarray) -> np.ndarray:
    x = np.asarray(window, dtype=np.float64)
    rms = np.sqrt(np.mean(x**2))
    peak = np.max(np.abs(x))
    crest_factor = peak / rms if rms > 1e-12 else 0.0
    return np.array(
        [
            x.mean(),
            x.std(),
            rms,
            peak,
            crest_factor,
            skew(x),
            kurtosis(x),
            x.max() - x.min(),  # peak-to-peak
        ]
    )


def spectral_features(window: np.ndarray, fs: float = 12000.0) -> np.ndarray:
    x = np.asarray(window, dtype=np.float64)
    n = len(x)
    spectrum = np.abs(rfft(x))
    freqs = rfftfreq(n, d=1.0 / fs)
    total_energy = spectrum.sum()
    if total_energy < 1e-12:
        centroid = 0.0
    else:
        centroid = float(np.sum(freqs * spectrum) / total_energy)
    peak_freq = float(freqs[np.argmax(spectrum)])
    spectral_std = float(np.sqrt(np.sum(((freqs - centroid) ** 2) * spectrum) / (total_energy + 1e-12)))
    spectral_energy = float(np.sum(spectrum**2))
    return np.array([centroid, peak_freq, spectral_std, spectral_energy])


def extract_features(window: np.ndarray, fs: float = 12000.0) -> np.ndarray:
    """Concatenate time-domain and spectral feature vectors for one window."""
    return np.concatenate([time_domain_features(window), spectral_features(window, fs)])


def extract_features_batch(X: np.ndarray, fs: float = 12000.0) -> np.ndarray:
    """Apply extract_features to each row of X, shape (N, window_size)."""
    return np.stack([extract_features(w, fs) for w in X])


FEATURE_NAMES = [
    "mean",
    "std",
    "rms",
    "peak",
    "crest_factor",
    "skewness",
    "kurtosis",
    "peak_to_peak",
    "spectral_centroid",
    "peak_frequency",
    "spectral_std",
    "spectral_energy",
]


In [ ]:
%%writefile src/models/__init__.py


In [ ]:
%%writefile src/models/classical.py
"""Classical supervised baselines on hand-crafted features."""
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC


def build_svm(seed: int = 0) -> SVC:
    return SVC(kernel="rbf", C=10.0, gamma="scale", random_state=seed)


def build_random_forest(seed: int = 0) -> RandomForestClassifier:
    return RandomForestClassifier(n_estimators=200, max_depth=None, random_state=seed, n_jobs=-1)


In [ ]:
%%writefile src/models/cnn1d.py
"""Small 1D-CNN backbone/classifier for raw vibration windows.

Kept small (<500K params) to run comfortably on CPU: 3 conv blocks with
increasing channels, global average pooling, small FC head.
"""
import torch
import torch.nn as nn


class CNN1DBackbone(nn.Module):
    """Feature extractor: raw window -> embedding vector."""

    def __init__(self, in_channels: int = 1, embedding_dim: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_channels, 16, kernel_size=32, stride=2, padding=15),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(16, 32, kernel_size=16, stride=2, padding=7),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(32, embedding_dim, kernel_size=8, stride=2, padding=3),
            nn.BatchNorm1d(embedding_dim),
            nn.ReLU(),
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.embedding_dim = embedding_dim

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, window_size) -> (B, 1, window_size)
        if x.dim() == 2:
            x = x.unsqueeze(1)
        z = self.net(x)
        z = self.pool(z).squeeze(-1)
        return z


class CNN1DClassifier(nn.Module):
    """Supervised classifier: backbone + linear head."""

    def __init__(self, n_classes: int, in_channels: int = 1, embedding_dim: int = 64):
        super().__init__()
        self.backbone = CNN1DBackbone(in_channels, embedding_dim)
        self.head = nn.Linear(embedding_dim, n_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        z = self.backbone(x)
        return self.head(z)


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


In [ ]:
%%writefile src/models/lstm.py
"""Shallow LSTM baseline for raw vibration windows."""
import torch
import torch.nn as nn


class ShallowLSTMClassifier(nn.Module):
    def __init__(
        self,
        n_classes: int,
        input_size: int = 1,
        hidden_size: int = 32,
        num_layers: int = 1,
        subsample: int = 4,
    ):
        """`subsample` downsamples the raw window before feeding the LSTM,
        keeping sequence length (and thus compute) small on CPU."""
        super().__init__()
        self.subsample = subsample
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )
        self.head = nn.Linear(hidden_size, n_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, window_size) -> (B, seq_len, 1)
        if x.dim() == 2:
            x = x[:, :: self.subsample].unsqueeze(-1)
        _, (h_n, _) = self.lstm(x)
        last_hidden = h_n[-1]
        return self.head(last_hidden)


In [ ]:
%%writefile src/models/prototypical.py
"""Prototypical Networks (Snell et al., 2017) for few-shot fault diagnosis.

Class prototypes are the mean embedding of the support set for each class;
query samples are classified by nearest prototype (negative squared
Euclidean distance as logits).
"""
from typing import Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F

from src.models.cnn1d import CNN1DBackbone


def compute_prototypes(
    support_embeddings: torch.Tensor, support_labels: torch.Tensor, n_classes: int
) -> torch.Tensor:
    """Return (n_classes, embedding_dim) prototype tensor."""
    embedding_dim = support_embeddings.shape[1]
    prototypes = torch.zeros(n_classes, embedding_dim, device=support_embeddings.device)
    for c in range(n_classes):
        mask = support_labels == c
        if mask.sum() > 0:
            prototypes[c] = support_embeddings[mask].mean(dim=0)
    return prototypes


def prototypical_logits(
    query_embeddings: torch.Tensor, prototypes: torch.Tensor
) -> torch.Tensor:
    """Negative squared Euclidean distance from each query to each prototype."""
    dists = torch.cdist(query_embeddings, prototypes, p=2) ** 2
    return -dists


class PrototypicalNetwork(nn.Module):
    def __init__(self, embedding_dim: int = 64, in_channels: int = 1):
        super().__init__()
        self.backbone = CNN1DBackbone(in_channels, embedding_dim)

    def embed(self, x: torch.Tensor) -> torch.Tensor:
        return self.backbone(x)

    def forward(
        self,
        support_x: torch.Tensor,
        support_y: torch.Tensor,
        query_x: torch.Tensor,
        n_classes: int,
    ) -> torch.Tensor:
        support_emb = self.embed(support_x)
        query_emb = self.embed(query_x)
        prototypes = compute_prototypes(support_emb, support_y, n_classes)
        return prototypical_logits(query_emb, prototypes)

    def classify_with_prototypes(
        self, support_x: torch.Tensor, support_y: torch.Tensor, query_x: torch.Tensor, n_classes: int
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Return (predicted_labels, logits) for query_x given the support set."""
        logits = self.forward(support_x, support_y, query_x, n_classes)
        preds = logits.argmax(dim=1)
        return preds, logits


def episodic_loss(logits: torch.Tensor, query_labels: torch.Tensor) -> torch.Tensor:
    return F.cross_entropy(logits, query_labels)


In [ ]:
%%writefile src/models/train_utils.py
"""Shared training loop helpers for torch-based models on CPU."""
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset


def make_loader(X: np.ndarray, y: np.ndarray, batch_size: int, shuffle: bool) -> DataLoader:
    X_t = torch.from_numpy(np.asarray(X, dtype=np.float32))
    y_t = torch.from_numpy(np.asarray(y, dtype=np.int64))
    return DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=shuffle)


def train_classifier(model, X_train, y_train, epochs, batch_size, lr, weight_decay=1e-4, device="cpu"):
    model.to(device)
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = torch.nn.CrossEntropyLoss()
    loader = make_loader(X_train, y_train, batch_size, shuffle=True)
    for _ in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
    return model


@torch.no_grad()
def predict(model, X, device="cpu", batch_size=256):
    model.eval()
    model.to(device)
    X_t = torch.from_numpy(np.asarray(X, dtype=np.float32))
    preds = []
    for i in range(0, len(X_t), batch_size):
        xb = X_t[i : i + batch_size].to(device)
        logits = model(xb)
        preds.append(logits.argmax(dim=1).cpu().numpy())
    return np.concatenate(preds)


In [ ]:
%%writefile src/models/episodic.py
"""Episodic training and evaluation for Prototypical Networks."""
from typing import Tuple

import numpy as np
import torch

from src.models.prototypical import PrototypicalNetwork, episodic_loss


def sample_episode(
    X: np.ndarray, y: np.ndarray, n_support: int, n_query: int, seed: int
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Sample a support/query split across all available classes in y."""
    rng = np.random.RandomState(seed)
    classes = np.unique(y)
    support_idx, query_idx = [], []
    for c in classes:
        cls_idx = np.where(y == c)[0]
        n_needed = n_support + n_query
        replace = len(cls_idx) < n_needed
        chosen = rng.choice(cls_idx, size=n_needed, replace=replace)
        support_idx.extend(chosen[:n_support])
        query_idx.extend(chosen[n_support:])
    support_idx = np.array(support_idx)
    query_idx = np.array(query_idx)
    return X[support_idx], y[support_idx], X[query_idx], y[query_idx]


def remap_labels(y: np.ndarray) -> Tuple[np.ndarray, dict]:
    classes = sorted(np.unique(y).tolist())
    mapping = {c: i for i, c in enumerate(classes)}
    y_mapped = np.array([mapping[v] for v in y], dtype=np.int64)
    return y_mapped, mapping


def train_protonet(
    model: PrototypicalNetwork,
    X_train: np.ndarray,
    y_train: np.ndarray,
    n_support: int,
    n_query: int,
    episodes_per_epoch: int,
    epochs: int,
    lr: float,
    seed: int,
    device: str = "cpu",
) -> PrototypicalNetwork:
    y_mapped, _ = remap_labels(y_train)
    n_classes = len(np.unique(y_mapped))
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()
        for ep_i in range(episodes_per_epoch):
            ep_seed = seed * 100000 + epoch * 1000 + ep_i
            sx, sy, qx, qy = sample_episode(X_train, y_mapped, n_support, n_query, ep_seed)
            sx_t = torch.from_numpy(sx.astype(np.float32)).to(device)
            sy_t = torch.from_numpy(sy).to(device)
            qx_t = torch.from_numpy(qx.astype(np.float32)).to(device)
            qy_t = torch.from_numpy(qy).to(device)

            optimizer.zero_grad()
            logits = model(sx_t, sy_t, qx_t, n_classes)
            loss = episodic_loss(logits, qy_t)
            loss.backward()
            optimizer.step()
    return model


@torch.no_grad()
def evaluate_protonet(
    model: PrototypicalNetwork,
    X_support: np.ndarray,
    y_support: np.ndarray,
    X_query: np.ndarray,
    device: str = "cpu",
) -> np.ndarray:
    """Classify X_query using prototypes computed from the full labeled
    (support) training set."""
    y_mapped, mapping = remap_labels(y_support)
    n_classes = len(mapping)
    inv_mapping = {v: k for k, v in mapping.items()}

    model.eval()
    sx_t = torch.from_numpy(X_support.astype(np.float32)).to(device)
    sy_t = torch.from_numpy(y_mapped).to(device)
    qx_t = torch.from_numpy(X_query.astype(np.float32)).to(device)

    preds, _ = model.classify_with_prototypes(sx_t, sy_t, qx_t, n_classes)
    preds_np = preds.cpu().numpy()
    return np.array([inv_mapping[p] for p in preds_np])


In [ ]:
%%writefile src/self_supervised/__init__.py


In [ ]:
%%writefile src/self_supervised/augmentations.py
"""Signal-specific augmentations for contrastive pretraining on vibration
windows: jitter, scaling, and time-warping."""
import numpy as np
import torch


def jitter(x: torch.Tensor, sigma: float = 0.03) -> torch.Tensor:
    return x + torch.randn_like(x) * sigma


def scaling(x: torch.Tensor, sigma: float = 0.1) -> torch.Tensor:
    factor = torch.randn(x.shape[0], 1, device=x.device) * sigma + 1.0
    return x * factor


def time_warp(x: torch.Tensor, n_knots: int = 4, sigma: float = 0.2) -> torch.Tensor:
    """Smoothly resample each signal along a randomly warped time axis
    using piecewise-linear interpolation on knot displacements."""
    batch_size, length = x.shape
    device = x.device
    out = torch.empty_like(x)
    orig_steps = np.linspace(0, length - 1, num=length)
    knot_positions = np.linspace(0, length - 1, num=n_knots + 2)
    for i in range(batch_size):
        warp = np.random.normal(loc=1.0, scale=sigma, size=n_knots + 2)
        warp[0] = warp[-1] = 1.0
        cum_warp = np.cumsum(warp)
        cum_warp = cum_warp / cum_warp[-1] * (length - 1)
        warped_steps = np.interp(orig_steps, knot_positions, cum_warp)
        signal_np = x[i].detach().cpu().numpy()
        out[i] = torch.from_numpy(np.interp(orig_steps, warped_steps, signal_np)).to(device)
    return out.float()


def augment_pair(x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Produce two augmented views of a batch for contrastive learning."""
    def view():
        v = jitter(x)
        v = scaling(v)
        v = time_warp(v)
        return v

    return view(), view()


In [ ]:
%%writefile src/self_supervised/simclr.py
"""SimCLR-style contrastive pretraining for 1D vibration signals.

Uses the NT-Xent (normalized temperature-scaled cross entropy) loss over
two augmented views per sample, with the CNN1DBackbone as encoder plus a
small MLP projection head (discarded after pretraining).
"""
import torch
import torch.nn as nn
import torch.nn.functional as F

from src.models.cnn1d import CNN1DBackbone


class ProjectionHead(nn.Module):
    def __init__(self, embedding_dim: int, hidden_dim: int = 64, out_dim: int = 32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class SimCLRModel(nn.Module):
    def __init__(self, embedding_dim: int = 64, projection_dim: int = 32):
        super().__init__()
        self.encoder = CNN1DBackbone(embedding_dim=embedding_dim)
        self.projector = ProjectionHead(embedding_dim, out_dim=projection_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.encoder(x)
        z = self.projector(h)
        return F.normalize(z, dim=1)


def nt_xent_loss(z1: torch.Tensor, z2: torch.Tensor, temperature: float = 0.5) -> torch.Tensor:
    """NT-Xent contrastive loss between two batches of L2-normalized
    embeddings, z1[i] and z2[i] being the positive pair."""
    batch_size = z1.shape[0]
    z = torch.cat([z1, z2], dim=0)  # (2B, D)
    sim = torch.mm(z, z.t()) / temperature  # (2B, 2B)

    mask = torch.eye(2 * batch_size, device=z.device, dtype=torch.bool)
    sim.masked_fill_(mask, float("-inf"))

    positive_idx = torch.cat(
        [torch.arange(batch_size, 2 * batch_size), torch.arange(0, batch_size)]
    ).to(z.device)

    return F.cross_entropy(sim, positive_idx)


In [ ]:
%%writefile src/self_supervised/masked_reconstruction.py
"""Masked-signal reconstruction pretext task (ablation vs. contrastive
pretraining). Randomly masks contiguous spans of the input signal and
trains an encoder-decoder to reconstruct the original values."""
import torch
import torch.nn as nn

from src.models.cnn1d import CNN1DBackbone


def mask_signal(x: torch.Tensor, mask_ratio: float = 0.15, span_len: int = 32) -> tuple[torch.Tensor, torch.Tensor]:
    """Zero out random contiguous spans of the signal. Returns (masked_x, mask)
    where mask is 1 where values were masked (and should be reconstructed)."""
    batch_size, length = x.shape
    mask = torch.zeros_like(x, dtype=torch.bool)
    n_spans = max(1, int((length * mask_ratio) / span_len))
    for i in range(batch_size):
        for _ in range(n_spans):
            start = torch.randint(0, max(1, length - span_len), (1,)).item()
            mask[i, start : start + span_len] = True
    masked_x = x.clone()
    masked_x[mask] = 0.0
    return masked_x, mask


class MaskedReconstructionModel(nn.Module):
    """Encoder (shared CNN1DBackbone) + simple decoder that upsamples the
    pooled embedding back to the original window length."""

    def __init__(self, window_size: int, embedding_dim: int = 64):
        super().__init__()
        self.encoder = CNN1DBackbone(embedding_dim=embedding_dim)
        self.decoder = nn.Sequential(
            nn.Linear(embedding_dim, 256),
            nn.ReLU(),
            nn.Linear(256, window_size),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        z = self.encoder(x)
        return self.decoder(z)


def reconstruction_loss(pred: torch.Tensor, target: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    """MSE computed only over masked positions."""
    diff = (pred - target) ** 2
    return (diff * mask).sum() / mask.sum().clamp(min=1)


In [ ]:
%%writefile src/evaluation/__init__.py


In [ ]:
%%writefile src/evaluation/metrics.py
"""Evaluation metrics: accuracy, macro-F1, confusion matrix."""
from typing import Dict

import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }


def compute_confusion_matrix(
    y_true: np.ndarray, y_pred: np.ndarray, n_classes: int
) -> np.ndarray:
    return confusion_matrix(y_true, y_pred, labels=list(range(n_classes)))


In [ ]:
%%writefile src/evaluation/significance.py
"""Statistical significance testing between methods across seeds."""
from typing import Dict, Tuple

import numpy as np
from scipy.stats import ttest_rel, wilcoxon


def paired_ttest(scores_a: np.ndarray, scores_b: np.ndarray) -> Tuple[float, float]:
    """Paired t-test between two methods' per-seed scores. Returns
    (statistic, p_value)."""
    scores_a = np.asarray(scores_a)
    scores_b = np.asarray(scores_b)
    if len(scores_a) != len(scores_b):
        raise ValueError("scores_a and scores_b must have the same length (paired seeds)")
    stat, p = ttest_rel(scores_a, scores_b)
    return float(stat), float(p)


def wilcoxon_signed_rank(scores_a: np.ndarray, scores_b: np.ndarray) -> Tuple[float, float]:
    scores_a = np.asarray(scores_a)
    scores_b = np.asarray(scores_b)
    if len(scores_a) != len(scores_b):
        raise ValueError("scores_a and scores_b must have the same length (paired seeds)")
    diffs = scores_a - scores_b
    if np.allclose(diffs, 0):
        return 0.0, 1.0
    stat, p = wilcoxon(scores_a, scores_b)
    return float(stat), float(p)


def summarize_across_seeds(scores: np.ndarray) -> Dict[str, float]:
    scores = np.asarray(scores)
    return {"mean": float(scores.mean()), "std": float(scores.std(ddof=1) if len(scores) > 1 else 0.0)}


In [ ]:
%%writefile src/evaluation/plots.py
"""Plotting utilities for results: accuracy vs. label fraction (central
figure), confusion matrices."""
from pathlib import Path
from typing import Dict, List

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns


def plot_accuracy_vs_label_fraction(
    results: Dict[str, Dict[float, Dict[str, float]]],
    metric: str = "accuracy",
    out_path: str | Path = "results/figures/accuracy_vs_label_fraction.png",
) -> None:
    """results: {method_name: {label_fraction: {"mean": .., "std": ..}}}"""
    fig, ax = plt.subplots(figsize=(7, 5))
    for method, per_fraction in results.items():
        fractions = sorted(per_fraction.keys())
        means = [per_fraction[f]["mean"] for f in fractions]
        stds = [per_fraction[f]["std"] for f in fractions]
        ax.errorbar(fractions, means, yerr=stds, marker="o", label=method, capsize=3)

    ax.set_xlabel("Label fraction")
    ax.set_ylabel(metric.replace("_", " ").title())
    ax.set_title(f"{metric.replace('_', ' ').title()} vs. Label Fraction")
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=200)
    plt.close(fig)


def plot_confusion_matrix(
    cm: np.ndarray,
    class_names: List[str],
    out_path: str | Path = "results/figures/confusion_matrix.png",
) -> None:
    fig, ax = plt.subplots(figsize=(8, 7))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=ax
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    fig.tight_layout()
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=200)
    plt.close(fig)


In [ ]:
%%writefile src/utils/__init__.py


In [ ]:
%%writefile src/utils/config.py
"""YAML config loading helpers."""
from pathlib import Path
from typing import Any, Dict

import yaml


def load_config(path: str | Path) -> Dict[str, Any]:
    with open(path, "r") as f:
        return yaml.safe_load(f)


In [ ]:
%%writefile src/utils/seeding.py
"""Deterministic seeding across numpy, random, and torch."""
import random

import numpy as np
import torch


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


In [ ]:
%%writefile tests/__init__.py


In [ ]:
%%writefile tests/test_preprocessing.py
import numpy as np
import pytest

from src.data.preprocessing import normalize_signal, sliding_window, windows_and_labels


def test_normalize_signal_zero_mean_unit_var():
    signal = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
    normed = normalize_signal(signal)
    assert abs(normed.mean()) < 1e-8
    assert abs(normed.std() - 1.0) < 1e-8


def test_normalize_signal_constant_input():
    signal = np.full(10, 5.0)
    normed = normalize_signal(signal)
    assert np.allclose(normed, 0.0)


def test_sliding_window_basic():
    signal = np.arange(10)
    windows = sliding_window(signal, window_size=4, stride=2)
    assert windows.shape == (4, 4)
    np.testing.assert_array_equal(windows[0], [0, 1, 2, 3])
    np.testing.assert_array_equal(windows[1], [2, 3, 4, 5])
    np.testing.assert_array_equal(windows[-1], [6, 7, 8, 9])


def test_sliding_window_no_overlap():
    signal = np.arange(12)
    windows = sliding_window(signal, window_size=4, stride=4)
    assert windows.shape == (3, 4)


def test_sliding_window_too_short_signal():
    signal = np.arange(3)
    windows = sliding_window(signal, window_size=4, stride=2)
    assert windows.shape == (0, 4)


def test_sliding_window_invalid_params():
    signal = np.arange(10)
    with pytest.raises(ValueError):
        sliding_window(signal, window_size=0, stride=2)
    with pytest.raises(ValueError):
        sliding_window(signal, window_size=4, stride=0)


def test_windows_and_labels_matches_shape():
    signal = np.arange(20)
    windows, labels = windows_and_labels(signal, label=3, window_size=5, stride=5)
    assert windows.shape[0] == labels.shape[0]
    assert np.all(labels == 3)


In [ ]:
%%writefile tests/test_metrics.py
import numpy as np

from src.evaluation.metrics import compute_confusion_matrix, compute_metrics
from src.evaluation.significance import paired_ttest, summarize_across_seeds, wilcoxon_signed_rank


def test_compute_metrics_perfect_prediction():
    y_true = np.array([0, 1, 2, 0, 1, 2])
    y_pred = np.array([0, 1, 2, 0, 1, 2])
    metrics = compute_metrics(y_true, y_pred)
    assert metrics["accuracy"] == 1.0
    assert metrics["f1_macro"] == 1.0


def test_compute_metrics_all_wrong():
    y_true = np.array([0, 0, 0])
    y_pred = np.array([1, 1, 1])
    metrics = compute_metrics(y_true, y_pred)
    assert metrics["accuracy"] == 0.0


def test_compute_confusion_matrix_shape():
    y_true = np.array([0, 1, 2, 1])
    y_pred = np.array([0, 1, 1, 1])
    cm = compute_confusion_matrix(y_true, y_pred, n_classes=3)
    assert cm.shape == (3, 3)
    assert cm.sum() == len(y_true)


def test_summarize_across_seeds():
    scores = np.array([0.8, 0.9, 0.85])
    stats = summarize_across_seeds(scores)
    assert abs(stats["mean"] - scores.mean()) < 1e-8
    assert stats["std"] >= 0


def test_paired_ttest_identical_scores_gives_high_pvalue():
    scores = np.array([0.8, 0.85, 0.9, 0.75])
    stat, p = paired_ttest(scores, scores.copy())
    assert p == 1.0 or np.isnan(p)


def test_paired_ttest_mismatched_lengths_raises():
    import pytest

    with pytest.raises(ValueError):
        paired_ttest(np.array([1, 2, 3]), np.array([1, 2]))


def test_wilcoxon_identical_scores():
    scores = np.array([0.8, 0.85, 0.9, 0.75])
    stat, p = wilcoxon_signed_rank(scores, scores.copy())
    assert p == 1.0


In [ ]:
%%writefile tests/test_splits.py
import numpy as np

from src.data.splits import cross_load_split, stratified_label_fraction_split


def test_stratified_label_fraction_split_full():
    y = np.array([0, 0, 1, 1, 2, 2])
    idx = stratified_label_fraction_split(y, fraction=1.0, seed=0)
    assert len(idx) == len(y)


def test_stratified_label_fraction_split_partial_preserves_all_classes():
    y = np.array([0] * 20 + [1] * 20 + [2] * 20)
    idx = stratified_label_fraction_split(y, fraction=0.1, seed=0)
    subset_labels = y[idx]
    assert set(np.unique(subset_labels)) == {0, 1, 2}
    assert len(idx) < len(y)


def test_stratified_label_fraction_split_deterministic():
    y = np.array([0] * 10 + [1] * 10)
    idx1 = stratified_label_fraction_split(y, fraction=0.3, seed=42)
    idx2 = stratified_label_fraction_split(y, fraction=0.3, seed=42)
    np.testing.assert_array_equal(idx1, idx2)


def test_cross_load_split_no_overlap():
    load_hp = np.array([0, 1, 2, 3, 0, 1, 2, 3])
    train_idx, test_idx = cross_load_split(load_hp, train_loads=[0, 1, 2], test_loads=[3])
    assert set(train_idx).isdisjoint(set(test_idx))
    assert all(load_hp[train_idx] != 3)
    assert all(load_hp[test_idx] == 3)


In [ ]:
%%writefile tests/test_features_and_models.py
import numpy as np
import torch

from src.features.handcrafted import extract_features, extract_features_batch, FEATURE_NAMES
from src.models.cnn1d import CNN1DBackbone, CNN1DClassifier, count_params
from src.models.prototypical import PrototypicalNetwork, compute_prototypes, prototypical_logits


def test_extract_features_length_matches_names():
    window = np.random.randn(1024)
    feats = extract_features(window, fs=12000.0)
    assert len(feats) == len(FEATURE_NAMES)
    assert np.all(np.isfinite(feats))


def test_extract_features_batch_shape():
    X = np.random.randn(5, 1024)
    feats = extract_features_batch(X)
    assert feats.shape == (5, len(FEATURE_NAMES))


def test_cnn1d_classifier_output_shape():
    model = CNN1DClassifier(n_classes=10, embedding_dim=32)
    x = torch.randn(4, 1024)
    out = model(x)
    assert out.shape == (4, 10)


def test_cnn1d_under_param_budget():
    model = CNN1DClassifier(n_classes=10, embedding_dim=64)
    assert count_params(model) < 500_000


def test_compute_prototypes_shape():
    embeddings = torch.randn(9, 8)
    labels = torch.tensor([0, 0, 0, 1, 1, 1, 2, 2, 2])
    protos = compute_prototypes(embeddings, labels, n_classes=3)
    assert protos.shape == (3, 8)


def test_prototypical_logits_shape():
    query = torch.randn(5, 8)
    protos = torch.randn(3, 8)
    logits = prototypical_logits(query, protos)
    assert logits.shape == (5, 3)


def test_protonet_forward_pass():
    model = PrototypicalNetwork(embedding_dim=16)
    support_x = torch.randn(6, 256)
    support_y = torch.tensor([0, 0, 1, 1, 2, 2])
    query_x = torch.randn(4, 256)
    logits = model(support_x, support_y, query_x, n_classes=3)
    assert logits.shape == (4, 3)


In [ ]:
%%writefile train.py
#!/usr/bin/env python
"""Train and evaluate a single method at a single label fraction / seed.

Usage:
    python train.py --config configs/default.yaml --method protonet \
        --label-fraction 0.1 --seed 0

Prints a JSON line with accuracy and f1_macro to stdout. Intended to be
called directly for debugging a single configuration, or invoked by
run_all_experiments.py to sweep the full grid.
"""
import argparse
import json

import numpy as np
from sklearn.preprocessing import StandardScaler

from src.data.splits import (
    cross_load_split,
    stratified_label_fraction_split,
    train_test_split_stratified,
)
from src.evaluation.metrics import compute_metrics
from src.features.handcrafted import extract_features_batch
from src.models.classical import build_random_forest, build_svm
from src.models.cnn1d import CNN1DClassifier
from src.models.episodic import evaluate_protonet, train_protonet
from src.models.lstm import ShallowLSTMClassifier
from src.models.prototypical import PrototypicalNetwork
from src.models.train_utils import predict, train_classifier
from src.self_supervised.simclr import SimCLRModel, nt_xent_loss
from src.self_supervised.augmentations import augment_pair
from src.utils.config import load_config
from src.utils.seeding import set_seed

import torch


def load_windows(processed_path: str):
    data = np.load(processed_path, allow_pickle=True)
    return data["X"], data["y"], data["load_hp"], list(data["class_names"])


def run_classical(method, X_train, y_train, X_test, y_test, seed, fs):
    feat_train = extract_features_batch(X_train, fs=fs)
    feat_test = extract_features_batch(X_test, fs=fs)
    if method == "svm":
        # RBF-SVM is scale-sensitive; our hand-crafted features mix
        # Hz-scale spectral values with unitless skew/kurtosis, so distances
        # are dominated by the largest-magnitude feature without scaling.
        scaler = StandardScaler()
        feat_train = scaler.fit_transform(feat_train)
        feat_test = scaler.transform(feat_test)
    model = build_svm(seed) if method == "svm" else build_random_forest(seed)
    model.fit(feat_train, y_train)
    y_pred = model.predict(feat_test)
    return compute_metrics(y_test, y_pred)


def run_cnn1d(X_train, y_train, X_test, y_test, n_classes, cfg, seed):
    model = CNN1DClassifier(n_classes=n_classes, embedding_dim=cfg["training"]["embedding_dim"])
    model = train_classifier(
        model, X_train, y_train,
        epochs=cfg["training"]["epochs"], batch_size=cfg["training"]["batch_size"],
        lr=cfg["training"]["learning_rate"], weight_decay=cfg["training"]["weight_decay"],
        device=cfg["training"]["device"],
    )
    y_pred = predict(model, X_test, device=cfg["training"]["device"])
    return compute_metrics(y_test, y_pred)


def run_lstm(X_train, y_train, X_test, y_test, n_classes, cfg, seed):
    model = ShallowLSTMClassifier(n_classes=n_classes)
    epochs = cfg.get("lstm", {}).get("epochs", cfg["training"]["epochs"])
    model = train_classifier(
        model, X_train, y_train,
        epochs=epochs, batch_size=cfg["training"]["batch_size"],
        lr=cfg["training"]["learning_rate"], weight_decay=cfg["training"]["weight_decay"],
        device=cfg["training"]["device"],
    )
    y_pred = predict(model, X_test, device=cfg["training"]["device"])
    return compute_metrics(y_test, y_pred)


def run_protonet(X_train, y_train, X_test, y_test, cfg, seed):
    pcfg = cfg["protonet"]
    n_support = min(pcfg["n_support"], min(np.bincount(y_train)) - 1) if len(np.unique(y_train)) > 0 else pcfg["n_support"]
    n_support = max(1, n_support)
    n_query = min(pcfg["n_query"], max(1, min(np.bincount(y_train)) - n_support))
    model = PrototypicalNetwork(embedding_dim=cfg["training"]["embedding_dim"])
    model = train_protonet(
        model, X_train, y_train, n_support=n_support, n_query=n_query,
        episodes_per_epoch=pcfg["episodes_per_epoch"], epochs=pcfg["epochs"],
        lr=cfg["training"]["learning_rate"], seed=seed, device=cfg["training"]["device"],
    )
    y_pred = evaluate_protonet(model, X_train, y_train, X_test, device=cfg["training"]["device"])
    return compute_metrics(y_test, y_pred)


def run_simclr_linear(X_unlabeled, X_train, y_train, X_test, y_test, n_classes, cfg, seed):
    scfg = cfg["simclr"]
    device = cfg["training"]["device"]
    model = SimCLRModel(embedding_dim=cfg["training"]["embedding_dim"], projection_dim=scfg["projection_dim"])
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["training"]["learning_rate"])

    X_unlabeled_t = torch.from_numpy(X_unlabeled.astype(np.float32))
    n = len(X_unlabeled_t)
    batch_size = scfg["batch_size"]
    for _ in range(scfg["epochs"]):
        perm = torch.randperm(n)
        for i in range(0, n, batch_size):
            idx = perm[i : i + batch_size]
            if len(idx) < 2:
                continue
            xb = X_unlabeled_t[idx].to(device)
            v1, v2 = augment_pair(xb)
            z1, z2 = model(v1), model(v2)
            loss = nt_xent_loss(z1, z2, temperature=scfg["temperature"])
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

    # Linear probe on frozen encoder using the (scarce) labeled data.
    # Precompute embeddings once (encoder is frozen) and train the probe
    # with mini-batches so the number of gradient steps scales with the
    # amount of labeled data, instead of a fixed handful of full-batch
    # updates that stayed the same regardless of label fraction.
    for p in model.encoder.parameters():
        p.requires_grad = False
    linear = torch.nn.Linear(cfg["training"]["embedding_dim"], n_classes).to(device)
    lin_opt = torch.optim.Adam(linear.parameters(), lr=cfg["training"]["learning_rate"])
    X_train_t = torch.from_numpy(X_train.astype(np.float32))
    y_train_t = torch.from_numpy(y_train.astype(np.int64))

    model.encoder.eval()
    with torch.no_grad():
        emb_train = model.encoder(X_train_t.to(device))
    probe_batch_size = min(scfg["batch_size"], len(emb_train))
    n_train = len(emb_train)
    for _ in range(scfg["linear_probe_epochs"]):
        perm = torch.randperm(n_train)
        for i in range(0, n_train, probe_batch_size):
            idx = perm[i : i + probe_batch_size]
            logits = linear(emb_train[idx])
            loss = torch.nn.functional.cross_entropy(logits, y_train_t[idx].to(device))
            lin_opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(linear.parameters(), max_norm=5.0)
            lin_opt.step()

    with torch.no_grad():
        X_test_t = torch.from_numpy(X_test.astype(np.float32)).to(device)
        emb_test = model.encoder(X_test_t)
        y_pred = linear(emb_test).argmax(dim=1).cpu().numpy()
    return compute_metrics(y_test, y_pred)


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--config", default="configs/default.yaml")
    parser.add_argument("--method", required=True)
    parser.add_argument("--label-fraction", type=float, required=True)
    parser.add_argument("--seed", type=int, required=True)
    args = parser.parse_args()

    cfg = load_config(args.config)
    set_seed(args.seed)

    X, y, load_hp, class_names = load_windows(cfg["data"]["processed_path"])
    n_classes = len(class_names)

    if cfg.get("cross_load", {}).get("enabled", False):
        train_idx, test_idx = cross_load_split(
            load_hp, cfg["cross_load"]["train_loads"], cfg["cross_load"]["test_loads"]
        )
        X_train_full, y_train_full = X[train_idx], y[train_idx]
        X_test, y_test = X[test_idx], y[test_idx]
    else:
        X_train_full, X_test, y_train_full, y_test = train_test_split_stratified(
            X, y, test_size=cfg["experiment"]["test_size"], seed=args.seed
        )

    sub_idx = stratified_label_fraction_split(y_train_full, args.label_fraction, args.seed)
    X_train, y_train = X_train_full[sub_idx], y_train_full[sub_idx]

    fs = cfg["data"]["sampling_rate_hz"]
    method = args.method

    if method in ("svm", "random_forest"):
        metrics = run_classical(method, X_train, y_train, X_test, y_test, args.seed, fs)
    elif method == "cnn1d":
        metrics = run_cnn1d(X_train, y_train, X_test, y_test, n_classes, cfg, args.seed)
    elif method == "lstm":
        metrics = run_lstm(X_train, y_train, X_test, y_test, n_classes, cfg, args.seed)
    elif method == "protonet":
        metrics = run_protonet(X_train, y_train, X_test, y_test, cfg, args.seed)
    elif method == "simclr_linear":
        metrics = run_simclr_linear(X_train_full, X_train, y_train, X_test, y_test, n_classes, cfg, args.seed)
    else:
        raise ValueError(f"Unknown method: {method}")

    result = {
        "method": method,
        "label_fraction": args.label_fraction,
        "seed": args.seed,
        **metrics,
    }
    print(json.dumps(result))
    return result


if __name__ == "__main__":
    main()


In [ ]:
%%writefile run_all_experiments.py
#!/usr/bin/env python
"""Run the full experiment grid: all methods x all label fractions x all
seeds, then aggregate results, run significance tests, and produce the
central accuracy-vs-label-fraction figure plus summary tables.

Results are checkpointed incrementally to a JSONL file as each run
completes, so the sweep can be safely interrupted and resumed: rerunning
with the same --config skips any (method, fraction, seed) combination
already present in the checkpoint file.

Usage:
    python run_all_experiments.py --config configs/default.yaml
    python run_all_experiments.py --config configs/default.yaml --aggregate-only
"""
import argparse
import json
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

from src.evaluation.plots import plot_accuracy_vs_label_fraction
from src.evaluation.significance import paired_ttest, summarize_across_seeds
from src.utils.config import load_config
from train import main as train_main
import sys


def checkpoint_path(out_cfg) -> Path:
    return Path(out_cfg["tables_dir"]) / "raw_results.jsonl"


def load_checkpoint(path: Path):
    records = []
    if path.exists():
        with open(path, "r") as f:
            for line in f:
                line = line.strip()
                if line:
                    records.append(json.loads(line))
    return records


def already_done(records):
    return {(r["method"], r["label_fraction"], r["seed"]) for r in records}


def run_grid(cfg, config_path, ckpt_path: Path):
    methods = cfg["experiment"]["methods"]
    fractions = cfg["experiment"]["label_fractions"]
    seeds = cfg["experiment"]["seeds"]

    records = load_checkpoint(ckpt_path)
    done = already_done(records)
    if done:
        print(f"Resuming: {len(done)} configurations already completed in {ckpt_path}")

    ckpt_path.parent.mkdir(parents=True, exist_ok=True)
    with open(ckpt_path, "a", buffering=1) as ckpt_f:
        for method in methods:
            for fraction in fractions:
                for seed in seeds:
                    key = (method, fraction, seed)
                    if key in done:
                        continue
                    print(f"=== method={method} fraction={fraction} seed={seed} ===", flush=True)
                    argv_backup = sys.argv
                    sys.argv = [
                        "train.py", "--config", config_path, "--method", method,
                        "--label-fraction", str(fraction), "--seed", str(seed),
                    ]
                    try:
                        result = train_main()
                        records.append(result)
                        ckpt_f.write(json.dumps(result) + "\n")
                        ckpt_f.flush()
                    except Exception as e:
                        print(f"FAILED: {method} fraction={fraction} seed={seed}: {e}", flush=True)
                    finally:
                        sys.argv = argv_backup
    return records


def aggregate(records, metric="accuracy"):
    grouped = defaultdict(lambda: defaultdict(list))
    for r in records:
        grouped[r["method"]][r["label_fraction"]].append(r[metric])

    summary = {}
    for method, per_fraction in grouped.items():
        summary[method] = {}
        for fraction, scores in per_fraction.items():
            summary[method][fraction] = summarize_across_seeds(np.array(scores))
    return summary, grouped


def run_significance_tests(grouped, metric="accuracy"):
    methods = list(grouped.keys())
    rows = []
    for i, m1 in enumerate(methods):
        for m2 in methods[i + 1 :]:
            common_fractions = set(grouped[m1].keys()) & set(grouped[m2].keys())
            for fraction in sorted(common_fractions):
                s1, s2 = grouped[m1][fraction], grouped[m2][fraction]
                if len(s1) == len(s2) and len(s1) > 1:
                    stat, p = paired_ttest(np.array(s1), np.array(s2))
                    rows.append(
                        {"method_a": m1, "method_b": m2, "label_fraction": fraction, "t_stat": stat, "p_value": p}
                    )
    return pd.DataFrame(rows)


def write_outputs(records, out_cfg):
    raw_path = Path(out_cfg["tables_dir"]) / "raw_results.json"
    with open(raw_path, "w") as f:
        json.dump(records, f, indent=2)
    print(f"Saved raw results to {raw_path}")

    for metric in ["accuracy", "f1_macro"]:
        summary, grouped = aggregate(records, metric=metric)
        plot_accuracy_vs_label_fraction(
            summary, metric=metric,
            out_path=Path(out_cfg["figures_dir"]) / f"{metric}_vs_label_fraction.png",
        )

        rows = []
        for method, per_fraction in summary.items():
            for fraction, stats in per_fraction.items():
                rows.append({"method": method, "label_fraction": fraction, **stats})
        df = pd.DataFrame(rows).sort_values(["method", "label_fraction"])
        table_path = Path(out_cfg["tables_dir"]) / f"{metric}_summary.csv"
        df.to_csv(table_path, index=False)
        print(f"Saved {metric} summary table to {table_path}")

        sig_df = run_significance_tests(grouped, metric=metric)
        sig_path = Path(out_cfg["tables_dir"]) / f"{metric}_significance.csv"
        sig_df.to_csv(sig_path, index=False)
        print(f"Saved {metric} significance tests to {sig_path}")


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--config", default="configs/default.yaml")
    parser.add_argument(
        "--aggregate-only", action="store_true",
        help="Skip training; just rebuild tables/figures from the existing checkpoint file.",
    )
    args = parser.parse_args()

    cfg = load_config(args.config)
    out_cfg = cfg["output"]
    Path(out_cfg["results_dir"]).mkdir(parents=True, exist_ok=True)
    Path(out_cfg["tables_dir"]).mkdir(parents=True, exist_ok=True)
    Path(out_cfg["figures_dir"]).mkdir(parents=True, exist_ok=True)

    ckpt_path = checkpoint_path(out_cfg)

    if args.aggregate_only:
        records = load_checkpoint(ckpt_path)
        print(f"Loaded {len(records)} checkpointed results from {ckpt_path}")
    else:
        records = run_grid(cfg, args.config, ckpt_path)
        print("Done.")

    write_outputs(records, out_cfg)


if __name__ == "__main__":
    main()


## 3. Install pinned dependencies

In [ ]:
!pip install -q -r requirements.txt


## 4. Download the CWRU Bearing Dataset

Public data, no credentials required — 12 kHz Drive-End fault subset (10 classes,
4 load conditions), served directly by the Case Western Reserve University Bearing
Data Center.

In [ ]:
!python -m src.data.download_cwru --out data/raw/cwru


## 5. Build windowed dataset (preprocessing + segmentation)

In [ ]:
!python -m src.data.build_dataset --raw data/raw/cwru --out data/processed --window-size 1024 --stride 512 --channel DE


## 6. Run unit tests (sanity check before the full sweep)

In [ ]:
!pytest tests/ -v


## 7. Run the full experiment grid

6 methods x 5 label fractions x 5 seeds = 150 runs. This is the long-running cell —
expect on the order of 1.5-3 hours on Colab's default CPU. Progress is checkpointed
incrementally to `results/tables/raw_results.jsonl`; if this cell is interrupted
(disconnect, runtime restart), just rerun it — completed configs are skipped
automatically.

In [ ]:
!python run_all_experiments.py --config configs/default.yaml


## 8. (Optional) Cross-load generalization experiment

Trains on CWRU loads 0-2 HP, evaluates on the held-out 3 HP condition.

In [ ]:
!python run_all_experiments.py --config configs/cross_load.yaml


## 9. View results

In [ ]:
import pandas as pd
from IPython.display import Image, display

acc_summary = pd.read_csv('results/tables/accuracy_summary.csv')
display(acc_summary)

f1_summary = pd.read_csv('results/tables/f1_macro_summary.csv')
display(f1_summary)

display(Image('results/figures/accuracy_vs_label_fraction.png'))
display(Image('results/figures/f1_macro_vs_label_fraction.png'))


In [ ]:
# Paired significance tests between methods at each label fraction
sig = pd.read_csv('results/tables/accuracy_significance.csv')
display(sig)


## 10. Download results locally

If you didn't mount Drive in step 0, zip and download the results folder here.

In [ ]:
!zip -rq results.zip results/
from google.colab import files
files.download('results.zip')
